In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np

import albumentations as A
from albumentations.pytorch import ToTensorV2

c:\Users\Santiago\Documents\Personal\IA\Numpy\pytorch\project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Hyperparameters
IMAGE_HEIGHT = 160
IMAGE_WIDTH = 240

In [5]:
class CovidDataset(Dataset):
  
  def __init__(self, images, masks, transform=None):
    self.images = images
    self.masks = masks
    self.transform = transform
    
  def __len__(self):
    return len(self.images)
  
  def _preprocess_mask(self, mask: np.ndarray) -> np.ndarray:
    mask = mask.astype(np.float32)
    if mask.max() > 1.0:
      mask = mask / 255.0
    return mask
    
  def __getitem__(self, index):
    image = self.images[index]
    mask = self._preprocess_mask(self.masks[index])
    
    if self.transform is not None:
      augmentations = self.transform(image=image, mask=mask)
      image = augmentations["image"]
      mask = augmentations["mask"]
      return image, mask
    

In [ ]:
class Transformer():
  def __init__(self, img_height=160, img_width=240, mean=None, std=None, max_pixel_value=None):
    self.img_height = img_height
    self.img_width = img_width
    self.mean=mean
    self.std=std
    self.max_pixel_value=max_pixel_value
    
  def train(self):
    return A.Compose([
      A.Resize(height=self.img_height, width=self.img_width),
      A.Rotate(limit=35, p=1.0),
      A.HorizontalFlip(p=0.5),
      A.VerticalFlip(p=0.1),
      A.Normalize(mean=self.mean, std=self.std, max_pixel_value=self.max_pixel_value),
      ToTensorV2()
    ])
    
  def validation(self):
    return A.Compose([
      A.Resize(height=self.img_height, width=self.img_width),
      A.Normalize(mean=self.mean, std=self.std, max_pixel_value=self.max_pixel_value),
      ToTensorV2()
    ])
    

In [ ]:
BATCH_SIZE = 32
transformer = Transformer()

train_images, train_masks = [], [] # Example - Use real data
val_images, val_masks = [], [] # Example

train_dataset = Dataset(train_images, train_masks, transformer.train())
val_dataset = Dataset(val_images, val_masks, transformer.validation())       

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)